# CVM-A1: phân tích runtime, peak RSS và sai số tương đối

Mặc định Run All bắt đầu một benchmark SCALE mới trên Ubuntu native bằng Python Runner, Generator và Core C++. Lượt đầy đủ có 270 trial ở 10 mốc và có thể kéo dài hàng chục phút. Chọn `QUICK_LIVE` để thử nhanh bằng Core thật hoặc chọn chế độ đọc khi cần xem dữ liệu đã lưu.

Sau lượt LIVE, TV4 xác thực cặp RAW/sidecar vừa tạo rồi xuất hình ghép và các bảng từ dữ liệu của lượt đó.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from urllib.parse import quote

import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc có pyproject.toml và src/.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_HOME = Path(os.environ.get("XDG_DATA_HOME", str(Path.home() / ".local" / "share")))
EXPORT_DIR = Path(
    os.environ.get(
        "BIG_O_EXPORT_DIR",
        str(DATA_HOME / "distinct-elements-cvm-analysis" / "exports"),
    )
).expanduser()


def current_environment() -> dict:
    compiler = shutil.which("g++") or shutil.which("clang++")
    compiler_version = "không tìm thấy GCC/Clang"
    if compiler:
        result = subprocess.run(
            [compiler, "--version"], capture_output=True, text=True, check=False
        )
        compiler_version = (result.stdout or result.stderr).splitlines()[0]
    packages = {}
    for distribution in ("numpy", "scipy", "matplotlib", "pandas", "jupyterlab", "ipykernel"):
        try:
            packages[distribution] = version(distribution)
        except PackageNotFoundError:
            packages[distribution] = "chưa cài"
    return {
        "kernel": sys.executable,
        "python": platform.python_version(),
        "os": platform.platform(),
        "compiler": compiler_version,
        "packages": packages,
    }


ENVIRONMENT = current_environment()
environment_lines = []
for key, value in ENVIRONMENT.items():
    formatted = json.dumps(value, ensure_ascii=False) if isinstance(value, dict) else value
    environment_lines.append(f"**{key}** `{formatted}`")
display(Markdown("\n".join(environment_lines)))

## Chọn chế độ

Chọn `MODE` trong ô cấu hình:

- `RUN_LIVE_SCALE` (mặc định): chạy mới cả 10 mốc SCALE, 270 trial, trong thư mục có run ID riêng.
- `QUICK_LIVE`: chạy thật 100k và 200k, hai trial cho mỗi workload, tổng cộng 12 trial.
- `READ_EXISTING_SCALE_FULL`: đọc bộ SCALE tối ưu đầy đủ từ manifest cấu hình và xác thực đủ 270 trial ở 10 mốc. Có thể đặt `BIG_O_SCALE_FULL_MANIFEST` để dùng manifest khác.
- `READ_EXISTING_P0`: đọc bộ dữ liệu Linux lịch sử theo manifest, xác thực 15 cặp RAW/sidecar và hiển thị hình ghép ở năm mốc P0.
- `READ_EXISTING_SCALE`: đọc RAW và sidecar của kế hoạch SCALE 1–2–5. Đặt `SCALE_RAW` trỏ tới CSV đã đo; có thể phân tích ngay khi hoàn tất một phần.
- `SMOKE_LINUX`: chạy preset kiểm tra pipeline thật qua API TV3, sau đó phân tích qua API TV4.
- `CUSTOM_SWEEP`: xem trước hoặc chạy cấu hình. Để chạy C++ thật, đặt `RUN_CUSTOM = True`; profile `TV4_P0` dùng ma trận trong kế hoạch dự án, profile `MANUAL` dùng các tham số tùy chỉnh.

Các phép đo mới chỉ chạy khi bộ kiểm tra xác nhận Ubuntu/Linux và không phát hiện WSL, máy ảo hoặc container.

In [ ]:
MODE = "RUN_LIVE_SCALE"
RUN_CUSTOM = False
SHOW_WORKLOAD_DETAILS = False  # True để xem và xuất thêm hình theo workload.
CUSTOM_PROFILE = "TV4_P0"  # TV4_P0 | MANUAL
READ_EXISTING_MANIFEST = ROOT / "notebooks" / "data" / "linux_p0" / "manifest.json"
SCALE_RAW = None  # Đường dẫn CSV cho chế độ READ_EXISTING_SCALE thủ công.
SCALE_FULL_MANIFEST = Path(
    os.environ.get(
        "BIG_O_SCALE_FULL_MANIFEST",
        str(ROOT / "notebooks" / "data" / "scale_full" / "manifest.json"),
    )
).expanduser()

# Chỉ dùng các giá trị này khi CUSTOM_PROFILE = "MANUAL".
CUSTOM_M = [256]
CUSTOM_N = 65536
CUSTOM_F0 = None
CUSTOM_EPSILON = 0.5
CUSTOM_DELTA = 0.05
CUSTOM_WORKLOADS = ["all_distinct"]
CUSTOM_SEED_GENERATOR = 44000
CUSTOM_SEED_ALGORITHM = 84000
CUSTOM_TRIALS = 2

ALLOWED_MODES = {
    "RUN_LIVE_SCALE", "QUICK_LIVE", "READ_EXISTING_SCALE_FULL",
    "READ_EXISTING_P0", "READ_EXISTING_SCALE",
    "SMOKE_LINUX", "CUSTOM_SWEEP",
}
if MODE not in ALLOWED_MODES:
    raise ValueError(f"MODE phải thuộc {sorted(ALLOWED_MODES)}, nhận được {MODE!r}.")
if CUSTOM_PROFILE not in {"TV4_P0", "MANUAL"}:
    raise ValueError("CUSTOM_PROFILE phải là TV4_P0 hoặc MANUAL.")


def validate_custom_config() -> None:
    u64_max = 2**64 - 1
    if not CUSTOM_M or any(
        isinstance(m, bool) or not isinstance(m, int) or not 1 <= m <= u64_max for m in CUSTOM_M
    ):
        raise ValueError("CUSTOM_M phải là danh sách số nguyên uint64 dương, không rỗng.")
    if len(set(CUSTOM_M)) != len(CUSTOM_M):
        raise ValueError("CUSTOM_M không được chứa giá trị lặp.")
    allowed_workloads = {"all_duplicate", "all_distinct", "uniform", "skewed", "zipf"}
    if not CUSTOM_WORKLOADS or len(set(CUSTOM_WORKLOADS)) != len(CUSTOM_WORKLOADS):
        raise ValueError("CUSTOM_WORKLOADS phải không rỗng và không lặp.")
    if any(workload not in allowed_workloads for workload in CUSTOM_WORKLOADS):
        raise ValueError(f"Workload phải thuộc {sorted(allowed_workloads)}.")
    for name, value in (("epsilon", CUSTOM_EPSILON), ("delta", CUSTOM_DELTA)):
        if not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 < value < 1:
            raise ValueError(f"{name} phải hữu hạn và thuộc (0, 1).")
    if isinstance(CUSTOM_TRIALS, bool) or not isinstance(CUSTOM_TRIALS, int) or CUSTOM_TRIALS < 1:
        raise ValueError("CUSTOM_TRIALS phải là số nguyên dương.")
    seeds = (CUSTOM_SEED_GENERATOR, CUSTOM_SEED_ALGORITHM)
    if any(
        isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed <= u64_max
        for seed in seeds
    ):
        raise ValueError("Seed phải thuộc miền uint64.")
    if max(seeds) + CUSTOM_TRIALS - 1 > u64_max:
        raise ValueError("Lịch seed vượt miền uint64 khi xét số trial.")
    if CUSTOM_N is not None and (
        isinstance(CUSTOM_N, bool) or not isinstance(CUSTOM_N, int) or not 1 <= CUSTOM_N <= 2**64
    ):
        raise ValueError("CUSTOM_N phải thuộc [1, 2**64] hoặc là None.")
    if CUSTOM_F0 is not None and (
        isinstance(CUSTOM_F0, bool)
        or not isinstance(CUSTOM_F0, int)
        or not 1 <= CUSTOM_F0 <= u64_max
    ):
        raise ValueError("CUSTOM_F0 phải là uint64 dương hoặc là None.")
    if CUSTOM_N is not None and CUSTOM_F0 is not None and CUSTOM_F0 > CUSTOM_N:
        raise ValueError("CUSTOM_F0 không được lớn hơn CUSTOM_N.")
    for m in CUSTOM_M:
        if CUSTOM_F0 is not None and CUSTOM_F0 > m:
            raise ValueError(f"F0={CUSTOM_F0} vượt m={m}.")
        for workload in CUSTOM_WORKLOADS:
            if workload == "all_distinct":
                if CUSTOM_N is not None and CUSTOM_N < m:
                    raise ValueError(f"all_distinct yêu cầu n >= m (m={m}, n={CUSTOM_N}).")
                if CUSTOM_F0 is not None and CUSTOM_F0 != m:
                    raise ValueError("all_distinct yêu cầu F0=m.")
            if workload == "all_duplicate" and CUSTOM_F0 not in (None, 1):
                raise ValueError("all_duplicate yêu cầu F0=1.")


if MODE == "CUSTOM_SWEEP" and CUSTOM_PROFILE == "MANUAL":
    validate_custom_config()

## Cơ sở lựa chọn ma trận P0

Notebook đọc ma trận từ [kế hoạch benchmark](../experiments/benchmark_plan_timespace.csv); các tham số không được khai báo lặp trong giao diện.

- `m` tăng theo hệ số 4, tạo các mức cách đều trên thang log₂ và bao phủ hệ số 256 từ mức nhỏ nhất đến lớn nhất.
- `n` bằng mức `m` lớn nhất để `all_distinct` có miền ID đủ rộng ở mọi cấu hình.
- `F₀` được xác định theo workload: `1` cho `all_duplicate`, `m/4` cho `uniform`, và `m` cho `all_distinct`. Vì vậy so sánh giữa workload phản ánh đồng thời phân bố và mức `F₀`; thiết kế này không tách riêng hai yếu tố.
- `ε` và `δ` giữ cố định theo giao thức P0. `δ` là tham số thất bại của thuật toán, không phải mức tin cậy của khoảng thống kê cho runtime. Ngưỡng `s` lấy từ hàm trong mã nguồn; tỷ số `F₀/s` được tính để xem tải của từng cấu hình so với ngưỡng.
- Chín lượt và lịch seed là một phần của giao thức P0. Kết quả được diễn giải mô tả; số lượt này không tự bảo đảm độ chính xác của ước lượng thống kê. Thứ tự cấu hình được xáo trộn bằng seed do bộ lập kế hoạch TV4 cung cấp.

## Kế hoạch SCALE (1–2–5)

[Kế hoạch mới](../experiments/benchmark_plan_timespace_scale.csv) có 10 mức m từ
100.000 đến 100.000.000. Dùng CLI để chọn một phần rồi đặt `MODE = "READ_EXISTING_SCALE"`
và `SCALE_RAW` để phân tích. Xem [hướng dẫn chạy từng phần](README.md#benchmark-scale-125).

In [ ]:
from scripts.generate_timespace_plan import ORDER_SEED
from src.types import compute_cvm_threshold


def tv4_p0_configurations() -> list[dict]:
    plan_path = ROOT / "experiments" / "benchmark_plan_timespace.csv"
    with plan_path.open(encoding="utf-8", newline="") as stream:
        rows = [row for row in csv.DictReader(stream) if row["priority"] == "P0"]
    if len(rows) != 15:
        raise ValueError(f"Kế hoạch P0 cần 15 cấu hình, nhận {len(rows)}.")

    configs = []
    for row in rows:
        config = {
            "workload": row["workload"],
            "m": int(row["m"]),
            "n": int(row["n"]),
            "f0": int(row["f0"]),
            "epsilon": float(row["epsilon"]),
            "delta": float(row["delta"]),
            "trials": int(row["trials"]),
            "first_trial": int(row["first_trial"]),
            "seed_generator": int(row["seed_generator"]),
            "seed_algorithm": int(row["seed_algorithm"]),
            "s": int(row["s_reference"]),
        }
        expected_f0 = {
            "all_duplicate": 1,
            "uniform": config["m"] // 4,
            "all_distinct": config["m"],
        }.get(config["workload"])
        if expected_f0 is None or config["f0"] != expected_f0:
            raise ValueError(f"F0 không khớp quy tắc workload: {config}.")
        if config["n"] < config["m"] or config["first_trial"] != 1:
            raise ValueError(f"Tham số P0 không hợp lệ: {config}.")
        if config["s"] != compute_cvm_threshold(config["m"], config["epsilon"], config["delta"]):
            raise ValueError(f"s_reference không khớp công thức mã nguồn: {config}.")
        configs.append(config)

    random.Random(ORDER_SEED).shuffle(configs)
    return configs

In [ ]:
from scripts.live_scale import require_native_ubuntu


def new_run_dir(label: str) -> Path:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = EXPORT_DIR / f"{label}-{timestamp}-{uuid.uuid4().hex[:8]}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def path_for_display(path: Path) -> str:
    resolved = path.expanduser().resolve()
    try:
        return str(resolved.relative_to(ROOT.resolve()))
    except ValueError:
        try:
            return "~/" + str(resolved.relative_to(Path.home().resolve()))
        except ValueError:
            return str(resolved)


def load_existing_manifest(manifest_path: Path) -> tuple[dict, list[dict]]:
    manifest_path = manifest_path.expanduser().resolve()
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy manifest P0: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != 1 or manifest.get("dataset_id") != "timespace_linux_p0":
        raise ValueError("Manifest không thuộc bộ timespace_linux_p0 được hỗ trợ.")
    if manifest.get("classification") != "verified_real":
        raise ValueError("Manifest P0 phải được phân loại verified_real.")
    inputs = manifest.get("inputs")
    if not isinstance(inputs, list) or len(inputs) != manifest.get("group_count"):
        raise ValueError("Danh sách RAW/sidecar trong manifest không hợp lệ.")
    if (
        manifest.get("record_count"),
        manifest.get("group_count"),
        manifest.get("expected_plots"),
    ) != (135, 15, 3):
        raise ValueError("Manifest P0 phải mô tả 135 record, 15 nhóm và ba hình.")
    native_checks = manifest.get("native_linux_check")
    required_checks = {"--vm: none", "--container: none"}
    if (
        not isinstance(native_checks, list)
        or not any(check.startswith("Ubuntu:") for check in native_checks)
        or not required_checks.issubset(native_checks)
    ):
        raise ValueError("Manifest thiếu bằng chứng Ubuntu và kiểm tra VM/container.")

    dataset_root = manifest_path.parent.resolve()
    checked_inputs = []
    total_records = 0
    configurations = set()
    for item in inputs:
        if not isinstance(item, dict):
            raise ValueError("Mỗi đầu vào trong manifest phải là một đối tượng JSON.")
        raw_path = (dataset_root / item["raw"]).resolve()
        sidecar_path = (dataset_root / item["sidecar"]).resolve()
        for path in (raw_path, sidecar_path):
            try:
                path.relative_to(dataset_root)
            except ValueError as error:
                raise ValueError(
                    "Đường dẫn manifest phải nằm bên trong thư mục dữ liệu."
                ) from error
            if not path.is_file():
                raise FileNotFoundError(f"Thiếu tệp trong manifest: {path}")
        raw_sha256 = hashlib.sha256(raw_path.read_bytes()).hexdigest()
        sidecar_sha256 = hashlib.sha256(sidecar_path.read_bytes()).hexdigest()
        if raw_sha256 != item.get("raw_sha256") or sidecar_sha256 != item.get("sidecar_sha256"):
            raise ValueError(f"Checksum không khớp với manifest: {raw_path.name}")
        metadata = json.loads(sidecar_path.read_text(encoding="utf-8"))
        environments = [
            str(batch.get("environment", {}).get("os", "")) for batch in metadata.get("batches", [])
        ]
        if not environments or any(not value.lower().startswith("linux") for value in environments):
            raise ValueError(f"Sidecar không ghi môi trường Linux: {sidecar_path}")
        if int(metadata.get("records_count", -1)) != int(item.get("records", -2)):
            raise ValueError(f"Số record trong sidecar không khớp manifest: {sidecar_path}")
        key = (item.get("workload"), int(item.get("m", 0)))
        if key in configurations:
            raise ValueError(f"Cấu hình trùng trong manifest: {key}")
        configurations.add(key)
        total_records += int(item.get("records", 0))
        checked_inputs.append(
            {
                "raw": raw_path,
                "metadata": sidecar_path,
                "configuration": item.get("configuration"),
            }
        )

    expected = {(config["workload"], config["m"]): config for config in tv4_p0_configurations()}
    if configurations != set(expected):
        raise ValueError("Manifest không chứa đúng 15 cấu hình trong kế hoạch P0.")
    for item in inputs:
        config = expected[(item["workload"], int(item["m"]))]
        expected_values = {
            "n": config["n"],
            "F0": config["f0"],
            "epsilon": config["epsilon"],
            "delta": config["delta"],
            "trials": config["trials"],
            "first_trial": config["first_trial"],
            "seed_generator": config["seed_generator"],
            "seed_algorithm": config["seed_algorithm"],
        }
        for key, expected_value in expected_values.items():
            if item.get(key) != expected_value:
                raise ValueError(f"Tham số {key} trong manifest không khớp kế hoạch P0.")
        if int(item.get("records", 0)) != config["trials"]:
            raise ValueError("Số record của cấu hình không khớp số trial trong kế hoạch P0.")
    if total_records != manifest["record_count"]:
        raise ValueError("Tổng số record trong manifest không khớp.")
    return manifest, checked_inputs


def export_input_pair(raw_path: Path, metadata_path: Path, output_dir: Path, index: int):
    # Fresh benchmark pairs already live under this run directory.
    run_dir = output_dir.parent
    if raw_path.is_relative_to(run_dir) and metadata_path.is_relative_to(run_dir):
        return raw_path, metadata_path
    # Copy archived inputs so the exported report travels with the verified data.
    target = output_dir / "inputs" / f"dataset-{index:02d}"
    target.mkdir(parents=True)
    raw_copy = target / raw_path.name
    metadata_copy = target / metadata_path.name
    shutil.copyfile(raw_path, raw_copy)
    shutil.copyfile(metadata_path, metadata_copy)
    return raw_copy, metadata_copy


def write_run_report(
    path: Path,
    label: str,
    provenance: dict,
    overview: list[str],
    aggregate_png: Path,
    workload_pngs: tuple[Path, ...],
) -> None:
    lines = [
        f"# CVM-A1 Time/Space · {label}",
        "",
        f"{provenance['record_count']} bản ghi · {provenance['group_count']} nhóm · "
        f"`{provenance['classification']}`",
        "",
        f"![Hình tổng hợp A–B–C]({quote(aggregate_png.name)})",
        "",
        "- [CSV tổng hợp theo m](aggregate_summary.csv)",
        "- [CSV chi tiết workload, RAW và run_id](benchmark_summary.csv)",
        "- [CSV TV4 theo cấu hình](summary.csv)",
        "- [Danh mục RAW và sidecar](raw_index.csv)",
        "- [Nguồn gốc dữ liệu và môi trường](provenance.json)",
        "",
        "## Nguồn dữ liệu",
        "",
    ]
    for index, entry in enumerate(provenance["inputs"], start=1):
        raw_link = quote(entry["raw"], safe="/")
        sidecar_link = quote(entry["sidecar"], safe="/")
        detail_link = quote(entry["report"], safe="/")
        lines.append(
            f"- Bộ {index} · {entry['records']} bản ghi · [RAW]({raw_link}) · "
            f"[Sidecar]({sidecar_link}) · [Báo cáo TV4]({detail_link})"
        )
    lines.extend(("", "## Môi trường đo", ""))
    for entry in provenance["measurement_environments"]:
        lines.append(
            f"- {entry['os']} · GCC/Clang `{entry['compiler']}` · "
            f"Python {entry['python']} · CPU {entry['cpu']}"
        )
    lines.extend(f"- {check}" for check in provenance["native_linux_check"] or ())
    lines.extend(("", "## Biểu đồ", ""))
    lines.extend(f"- {note}" for note in overview)
    lines.extend(
        (
            "- Runtime thuật toán tính theo ms. Peak RSS là bộ nhớ thường trú cực đại "
            "của tiến trình C++. Bảng số liệu biểu đồ dùng MiB, summary.csv giữ byte và "
            "kích thước tập mẫu.",
            "- Mỗi điểm là trung bình không trọng số của các median workload có dữ liệu. "
            "Đây là số tổng hợp từ các lượt đo Core.",
            "- Relative Error (%) được tính từ estimate và F₀ của từng lượt success có F₀ > 0, "
            "trước khi lấy median theo workload. CSV ghi độ bao phủ và số trial theo từng metric.",
            "- Các điểm đo mô tả cấu hình đã chạy. Kết luận Big-O cần phân tích "
            "chi phí thuật toán.",
        )
    )
    for png_path in workload_pngs:
        lines.append(f"- [Hình chi tiết workload ({png_path.stem})]({quote(png_path.name)})")
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def analyze_datasets(
    datasets: list[dict],
    output_dir: Path,
    *,
    label: str,
    native_evidence: list[str] | None = None,
    expected_records: int | None = None,
    expected_groups: int | None = None,
    source_manifest: Path | None = None,
    source_plan: Path | None = None,
    show_workload_details: bool = False,
) -> dict:
    from scripts.analyze_results import (
        aggregate_by_length,
        benchmark_rows,
        summarize,
        summary_rows,
        write_summary,
    )
    from scripts.generate_timespace_plan import BASE_WORKLOADS, validate_scale_rows
    from scripts.jupyter_analysis import analyze_jupyter_results
    from scripts.plot_results import plot_aggregate, plot_summary
    from scripts.validate_results import COMPLETED, completed_trial_key

    if not datasets:
        raise ValueError("Không có RAW/sidecar để phân tích.")
    output_dir = output_dir.resolve()
    raw_frames, provenance_rows, reports = [], [], []
    for index, item in enumerate(datasets, start=1):
        raw_path = Path(item["raw"]).expanduser().resolve()
        metadata_path = Path(item["metadata"]).expanduser().resolve()
        raw_path, metadata_path = export_input_pair(raw_path, metadata_path, output_dir, index)
        report = analyze_jupyter_results(
            raw_path, output_dir / "datasets", metadata_path=metadata_path, plot_workloads=False
        )
        expected_hashes = item.get("expected_hashes")
        if expected_hashes is not None and expected_hashes != (
            report.provenance.raw_sha256,
            report.provenance.metadata_sha256,
        ):
            raise ValueError("RAW/sidecar đã thay đổi sau khi xác thực manifest.")
        reports.append(report)
        raw_frames.append(report.raw)
        provenance_rows.append(
            {
                "configuration": item.get("configuration", f"dataset-{index:02d}"),
                "raw": os.path.relpath(report.provenance.raw_path, output_dir),
                "sidecar": os.path.relpath(report.provenance.metadata_path, output_dir),
                "report": os.path.relpath(report.markdown_path, output_dir),
                "records": len(report.raw),
                "classification": report.provenance.classification,
                "raw_sha256": report.provenance.raw_sha256,
                "sidecar_sha256": report.provenance.metadata_sha256,
            }
        )

    raw = pd.concat(raw_frames, ignore_index=True)
    if raw["run_id"].duplicated().any():
        raise ValueError("Phát hiện run_id trùng giữa các RAW đã xác thực.")
    verified = [row for report in reports for row in report.verified_rows]
    completed = [completed_trial_key(row) for row in verified if row["status"] in COMPLETED]
    if len(completed) != len(set(completed)):
        raise ValueError("Phát hiện trial đã hoàn tất trùng giữa các RAW đã xác thực.")
    if source_plan is not None:
        validate_scale_rows(verified)

    summary_groups = summarize(verified, include_trials=True)
    historical_summary = summary_rows(summary_groups)
    summary = pd.DataFrame.from_records(historical_summary)
    summary["s"] = pd.Series([group["s"] for group in summary_groups], dtype=object)
    benchmark = benchmark_rows(summary_groups)
    sources_by_run = {
        run_id: entry["raw"]
        for report, entry in zip(reports, provenance_rows, strict=True)
        for run_id in report.raw["run_id"]
    }
    for row in benchmark:
        paths = sorted({sources_by_run[run_id] for run_id in row["run_ids"].split()})
        row["raw_sources"] = json.dumps(paths, ensure_ascii=False)
    aggregate = aggregate_by_length(summary_groups)
    if expected_records is not None and len(raw) != expected_records:
        raise ValueError(f"Cần {expected_records} record, nhận {len(raw)}.")
    if expected_groups is not None and len(summary) != expected_groups:
        raise ValueError(f"Cần {expected_groups} nhóm, nhận {len(summary)}.")

    summary_path = output_dir / "summary.csv"
    benchmark_path = output_dir / "benchmark_summary.csv"
    aggregate_path = output_dir / "aggregate_summary.csv"
    index_path = output_dir / "raw_index.csv"
    manifest_path = output_dir / "provenance.json"
    report_path = output_dir / "report.md"
    write_summary(summary_path, historical_summary)
    write_summary(benchmark_path, benchmark)
    write_summary(aggregate_path, aggregate)
    index_frame = pd.DataFrame(provenance_rows)
    index_frame.to_csv(index_path, index=False)
    environment_fields = ("os", "python", "cpu", "compiler", "build_flags")
    environments = dict.fromkeys(
        tuple(batch["environment"].get(key, "—") for key in environment_fields)
        for report in reports
        for batch in report.provenance.metadata["batches"]
    )
    environment_rows = [
        dict(zip(environment_fields, values, strict=True)) for values in environments
    ]
    aggregate_png = plot_aggregate(aggregate, output_dir, label=label)
    workload_pngs = (
        tuple(plot_summary(summary_groups, output_dir, label=label))
        if show_workload_details
        else ()
    )
    overview = []
    mixed_sizes = [
        row["m"] for row in aggregate
        if any(value == "mixed_configuration" for value in row.values())
    ]
    if mixed_sizes:
        sizes = ", ".join(f"{m:,}" for m in mixed_sizes)
        overview.append(
            f"Có chỉ số bỏ trống tại m = {sizes} do cấu hình hoặc môi trường đo khác nhau. "
            "Xem cột status trong CSV tổng hợp và bảng chi tiết."
        )
    if not any(row["runtime_ms"] is not None for row in aggregate):
        overview.append("Chưa có điểm runtime đủ điều kiện tổng hợp trong ba workload chính.")
    if sum(row["runtime_ms"] is not None for row in aggregate) == 1:
        overview.append("Chỉ có một điểm runtime. Chưa đủ điểm để nhận xét xu hướng.")

    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "analysis_api": "scripts.jupyter_analysis.analyze_jupyter_results",
        "plot_api": "scripts.plot_results.plot_aggregate",
        "aggregation": {
            "method": "unweighted_mean_of_workload_medians",
            "workloads": list(BASE_WORKLOADS),
            "relative_error": "abs(estimate-F0)/F0*100 per successful trial with F0>0",
        },
        "classification": "verified_real",
        "kernel": sys.executable,
        "python": platform.python_version(),
        "kernel_os": platform.platform(),
        "packages": ENVIRONMENT["packages"],
        "notebook_sha256": hashlib.sha256(
            (ROOT / "notebooks" / "BIG_O_TimeSpace.ipynb").read_bytes()
        ).hexdigest(),
        "source_manifest": (
            {
                "path": path_for_display(source_manifest),
                "sha256": hashlib.sha256(source_manifest.read_bytes()).hexdigest(),
            }
            if source_manifest is not None
            else None
        ),
        "source_plan": (
            {
                "path": path_for_display(source_plan),
                "sha256": hashlib.sha256(source_plan.read_bytes()).hexdigest(),
            }
            if source_plan is not None
            else None
        ),
        "benchmark_summary": {
            "path": benchmark_path.name,
            "sha256": hashlib.sha256(benchmark_path.read_bytes()).hexdigest(),
        },
        "aggregate_summary": {
            "path": aggregate_path.name,
            "sha256": hashlib.sha256(aggregate_path.read_bytes()).hexdigest(),
        },
        "pngs": [aggregate_png.name],
        "combined_pngs": [aggregate_png.name],
        "workload_pngs": [path.name for path in workload_pngs],
        "native_linux_check": native_evidence,
        "input_count": len(provenance_rows),
        "record_count": len(raw),
        "group_count": len(summary),
        "inputs": provenance_rows,
        "measurement_environments": environment_rows,
    }
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    write_run_report(report_path, label, manifest, overview, aggregate_png, workload_pngs)

    aggregate_frame = pd.DataFrame.from_records(aggregate)
    display_summary = aggregate_frame[["m", "runtime_ms", "peak_ram_mib", "relative_error_pct"]].rename(
        columns={
            "m": "Input size m",
            "runtime_ms": "Runtime (ms)",
            "peak_ram_mib": "Peak RAM (MiB)",
            "relative_error_pct": "Relative Error (%)",
        }
    )
    for field, title in (
        ("runtime_ms", "Runtime"),
        ("peak_ram_mib", "RAM"),
        ("relative_error_pct", "Error"),
    ):
        display_summary[f"{title} · workload / trial"] = [
            f"{row[f'{field}_workloads']}/{len(BASE_WORKLOADS)} · {row[f'{field}_trials']}"
            for row in aggregate
        ]
    display_summary = display_summary.astype(object).where(display_summary.notna(), "N/A")
    benchmark_frame = pd.DataFrame.from_records(benchmark)
    benchmark_frame["s"] = pd.Series([row["s"] for row in benchmark], dtype=object)

    display(Markdown(f"## Kết quả · {label}"))
    display(Image(filename=str(aggregate_png), width=850))
    display(display_summary)
    display(Markdown("\n".join(
        f"- **{title}** `{path_for_display(path)}`"
        for title, path in (
            ("CSV tổng hợp theo m", aggregate_path),
            ("CSV chi tiết workload và RAW", benchmark_path),
            ("PNG ghép A–B–C", aggregate_png),
            ("Báo cáo", report_path),
            ("Provenance", manifest_path),
        )
    )))
    for note in overview:
        display(Markdown(f"- {note}"))
    if show_workload_details:
        display(Markdown("### Chi tiết theo workload"))
        detail_columns = (
            "m workload F0 s f0_over_m runtime_median_ms peak_rss_median_mib "
            "relative_error_median_pct success_count raw_sources"
        ).split()
        detail = benchmark_frame[detail_columns].astype(object)
        display(detail.where(detail.notna(), "N/A"))
        for png_path in workload_pngs:
            display(Image(filename=str(png_path), width=900))
    return {
        "raw": raw,
        "summary": summary,
        "benchmark_summary": benchmark_frame,
        "aggregate_summary": aggregate_frame,
        "png_paths": (aggregate_png,),
        "combined_png_paths": (aggregate_png,),
        "workload_pngs": workload_pngs,
        "provenance": provenance_rows,
    }


def show_benchmark_result(result) -> None:
    display(
        Markdown(
            f"**API TV3** · {result.records_count} bản ghi · {result.configs_count} cấu hình\n\n"
            f"**RAW** `{path_for_display(result.csv_path)}`\n\n"
            f"**Sidecar** `{path_for_display(result.sidecar_path)}`"
        )
    )

In [ ]:
analysis_result = None
benchmark_results = []

if MODE in {"RUN_LIVE_SCALE", "QUICK_LIVE"}:
    from scripts.generate_timespace_plan import SCALE_PLAN
    from scripts.live_scale import EVENTS_NAME, GIB, RAW_NAME, STATUS_NAME, run_live_scale

    live_dir = new_run_dir("scale-live" if MODE == "RUN_LIVE_SCALE" else "quick-live")
    display(Markdown(f"**{MODE}** · Thư mục kết quả mới `{path_for_display(live_dir)}`"))
    handle = display(Markdown("Đang khởi động bộ giám sát…"), display_id=True)
    progress = {"success": 0, "trial": 0}

    def show_live_event(event):
        kind = event["event"]
        if kind == "run_started":
            progress["start"] = event["monotonic"]
            progress["total"] = event["total_trials"]
        elif kind == "batch_started":
            progress.update(m=event["m"], workload=event["workload"], trial=0,
                            batch_trials=event["trials"])
        elif kind == "trial_started":
            progress.update(m=event["m"], workload=event["workload"], trial=event["trial"])
        elif kind == "trial_finished" and event["status"] == "success":
            progress["success"] += 1
        elif kind == "resource_sample":
            progress.update(rss=event["pipeline_rss_bytes"], ram=event["available_bytes"],
                            swap=event["swap_growth_bytes"])
        elif kind == "run_finished":
            progress["success"] = event["successful_trials"]
        else:
            return
        elapsed = event["monotonic"] - progress["start"]
        line = f"**{progress['success']}/{progress['total']} trial thành công** · {elapsed:.1f} s"
        if "m" in progress:
            trial = progress["trial"] or "chuẩn bị"
            line += (
                f" · m={progress['m']:,} · `{progress['workload']}`"
                f" · trial {trial}/{progress.get('batch_trials', '?')}"
            )
        if "rss" in progress:
            line += (
                f"\n\nRSS cây tiến trình {progress['rss'] / GIB:.3f} GiB"
                f" · RAM khả dụng {progress['ram'] / GIB:.2f} GiB"
                f" · Swap tăng {progress['swap'] / GIB:.3f} GiB"
            )
        handle.update(Markdown(line))

    live_result = run_live_scale(live_dir, MODE, show_live_event)
    live_status = {"complete": "Hoàn tất", "interrupted": "Đã dừng", "failed": "Thất bại"}
    status_text = live_status[live_result["status"]]
    display(Markdown(
        f"**{status_text}** · **{live_result['successful_trials']}/{live_result['planned_trials']}**"
        f" trial thành công. {live_result.get('reason', '')}"
    ))
    retained_paths = (
        ("RAW", RAW_NAME), ("Sidecar", "scale-live.metadata.json"),
        ("Sự kiện LIVE", EVENTS_NAME), ("Provenance", STATUS_NAME),
        ("Log supervisor", "supervisor.log"), ("Log batch", "logs"),
    )
    display(Markdown("\n".join(
        f"- {title} `{path_for_display(live_dir / name)}`"
        for title, name in retained_paths
        if (live_dir / name).exists()
    )))
    if live_result.get("raw_verified") and live_result["successful_trials"]:
        raw_path = live_dir / RAW_NAME
        analysis_result = analyze_datasets(
            [{
                "raw": raw_path,
                "metadata": raw_path.with_suffix(".metadata.json"),
                "configuration": MODE,
                "expected_hashes": (live_result["raw_sha256"], live_result["sidecar_sha256"]),
            }],
            live_dir / "analysis",
            label=f"CVM-A1 · {MODE} · {status_text}",
            native_evidence=live_result["native_linux_check"],
            expected_records=live_result["recorded_trials"],
            source_manifest=live_dir / STATUS_NAME,
            source_plan=SCALE_PLAN,
            show_workload_details=SHOW_WORKLOAD_DETAILS,
        )

elif MODE == "READ_EXISTING_SCALE_FULL":
    from scripts.generate_timespace_plan import SCALE_PLAN
    from scripts.jupyter_analysis import load_scale_full_manifest

    source_manifest = SCALE_FULL_MANIFEST.resolve()
    scale_manifest, dataset = load_scale_full_manifest(source_manifest)
    label = scale_manifest["name"]
    display(
        Markdown(
            f"**{label}** · **{dataset['record_count']} trial hợp lệ**"
            f" · **{dataset['size_count']} mốc SCALE**"
            f" · {dataset['group_count']} cấu hình\n\n"
            f"Pipeline SHA-256 `{scale_manifest['source_tree_sha256']}`"
        )
    )
    run_dir = new_run_dir("linux-scale-full-analysis")
    analysis_result = analyze_datasets(
        [dataset],
        run_dir / "analysis",
        label=label,
        native_evidence=scale_manifest.get("native_linux_check"),
        expected_records=dataset["record_count"],
        expected_groups=dataset["group_count"],
        source_manifest=source_manifest,
        source_plan=SCALE_PLAN,
        show_workload_details=SHOW_WORKLOAD_DETAILS,
    )

elif MODE == "READ_EXISTING_P0":
    source_manifest = READ_EXISTING_MANIFEST.expanduser().resolve()
    p0_manifest, checked_inputs = load_existing_manifest(source_manifest)
    display(
        Markdown(
            "Đang phân tích dữ liệu P0 Linux đã lưu.\n\n**Môi trường đo ghi trong manifest**\n\n"
            + "\n".join(f"- {value}" for value in p0_manifest["native_linux_check"])
        )
    )
    run_dir = new_run_dir("linux-p0-analysis")
    analysis_result = analyze_datasets(
        checked_inputs,
        run_dir / "analysis",
        label="P0 Linux đã lưu",
        native_evidence=p0_manifest.get("native_linux_check"),
        expected_records=p0_manifest["record_count"],
        expected_groups=p0_manifest["group_count"],
        source_manifest=source_manifest,
        show_workload_details=SHOW_WORKLOAD_DETAILS,
    )

elif MODE == "READ_EXISTING_SCALE":
    from scripts.generate_timespace_plan import SCALE_PLAN

    if SCALE_RAW is None:
        raise ValueError("Đặt SCALE_RAW trỏ tới CSV đã đo của kế hoạch SCALE.")
    raw_path = (ROOT / Path(SCALE_RAW).expanduser()).resolve()
    run_dir = new_run_dir("linux-scale-analysis")
    analysis_result = analyze_datasets(
        [
            {
                "raw": raw_path,
                "metadata": raw_path.with_suffix(".metadata.json"),
                "configuration": "SCALE",
            }
        ],
        run_dir / "analysis",
        label="SCALE (1–2–5)",
        show_workload_details=SHOW_WORKLOAD_DETAILS,
        source_plan=SCALE_PLAN,
    )

elif MODE == "SMOKE_LINUX":
    native_evidence = require_native_ubuntu()
    display(
        Markdown(
            "**Xác nhận môi trường:**\n\n" + "\n".join(f"- {item}" for item in native_evidence)
        )
    )
    from src.jupyter_adapter import run_jupyter_benchmark

    run_dir = new_run_dir("smoke-linux")
    result = run_jupyter_benchmark(preset="smoke", output_dir=run_dir / "real", verbose=False)
    benchmark_results.append(result)
    show_benchmark_result(result)
    analysis_result = analyze_datasets(
        [{"raw": result.csv_path, "metadata": result.sidecar_path, "configuration": "SMOKE_LINUX"}],
        run_dir / "analysis",
        label="SMOKE_LINUX",
        show_workload_details=SHOW_WORKLOAD_DETAILS,
        native_evidence=native_evidence,
    )

elif MODE == "CUSTOM_SWEEP":
    if not RUN_CUSTOM:
        if CUSTOM_PROFILE == "TV4_P0":
            plan = tv4_p0_configurations()
        else:
            plan = [
                {
                    "m": m,
                    "n": CUSTOM_N,
                    "f0": CUSTOM_F0,
                    "workload": workload,
                    "epsilon": CUSTOM_EPSILON,
                    "delta": CUSTOM_DELTA,
                    "trials": CUSTOM_TRIALS,
                    "seed_generator": CUSTOM_SEED_GENERATOR,
                    "seed_algorithm": CUSTOM_SEED_ALGORITHM,
                }
                for workload in CUSTOM_WORKLOADS
                for m in CUSTOM_M
            ]
        attempts = sum(config["trials"] for config in plan)
        display(
            Markdown(
                f"**Xem trước** · {len(plan)} cấu hình · {attempts} lượt dự kiến. "
                "Đặt `RUN_CUSTOM = True` để chạy benchmark trên Ubuntu native."
            )
        )
        preview = pd.DataFrame(plan)
        if CUSTOM_PROFILE == "TV4_P0":
            preview["F0_over_s"] = preview["f0"] / preview["s"]
        display(preview)
    else:
        native_evidence = require_native_ubuntu()
        display(
            Markdown(
                "**Xác nhận môi trường:**\n\n" + "\n".join(f"- {item}" for item in native_evidence)
            )
        )
        from src.jupyter_adapter import run_jupyter_benchmark

        run_dir = new_run_dir("linux-p0" if CUSTOM_PROFILE == "TV4_P0" else "custom-linux")
        if CUSTOM_PROFILE == "TV4_P0":
            datasets = []
            configs = tv4_p0_configurations()
            for index, config in enumerate(configs, start=1):
                tag = f"{config['workload']}_m{config['m']}"
                print(
                    f"TV3 cấu hình {index}/{len(configs)} · {tag} · "
                    f"F0={config['f0']} · {config['trials']} trial",
                    flush=True,
                )
                result = run_jupyter_benchmark(
                    m=config["m"],
                    n=config["n"],
                    f0=config["f0"],
                    epsilon=config["epsilon"],
                    delta=config["delta"],
                    workload=config["workload"],
                    trials=config["trials"],
                    seed_generator=config["seed_generator"],
                    seed_algorithm=config["seed_algorithm"],
                    output_dir=run_dir / "real",
                    tag=tag,
                    verbose=False,
                )
                benchmark_results.append(result)
                datasets.append(
                    {"raw": result.csv_path, "metadata": result.sidecar_path, "configuration": tag}
                )
            total_records = sum(item.records_count for item in benchmark_results)
            display(
                Markdown(
                    f"**API TV3** · {total_records} bản ghi · {len(configs)} cấu hình "
                    f"· {len(benchmark_results)} cặp RAW/sidecar"
                )
            )
            analysis_result = analyze_datasets(
                datasets,
                run_dir / "analysis",
                label="Linux P0",
                show_workload_details=SHOW_WORKLOAD_DETAILS,
                native_evidence=native_evidence,
                expected_records=135,
                expected_groups=15,
            )
        else:
            result = run_jupyter_benchmark(
                m=CUSTOM_M,
                n=CUSTOM_N,
                f0=CUSTOM_F0,
                epsilon=CUSTOM_EPSILON,
                delta=CUSTOM_DELTA,
                workload=CUSTOM_WORKLOADS,
                trials=CUSTOM_TRIALS,
                seed_generator=CUSTOM_SEED_GENERATOR,
                seed_algorithm=CUSTOM_SEED_ALGORITHM,
                output_dir=run_dir / "real",
                tag="custom_sweep",
                verbose=False,
            )
            benchmark_results.append(result)
            show_benchmark_result(result)
            analysis_result = analyze_datasets(
                [
                    {
                        "raw": result.csv_path,
                        "metadata": result.sidecar_path,
                        "configuration": "MANUAL_SWEEP",
                    }
                ],
                run_dir / "analysis",
                label="CUSTOM_SWEEP",
                show_workload_details=SHOW_WORKLOAD_DETAILS,
                native_evidence=native_evidence,
            )

## Diễn giải và giới hạn

RAW và sidecar được giữ nguyên theo từng cấu hình. TV4 xác thực từng cặp rồi tính median theo workload qua các lượt success có số đo. Hình ghép dùng trung bình không trọng số của các median đó cho runtime lõi, peak RSS tiến trình C++ và sai số tương đối. Bảng chính ghi số workload và số trial hợp lệ riêng cho mỗi metric. Peak sample có trong summary.csv. RSS không bao gồm bộ nhớ của Python/Jupyter.

Các bảng và hình mô tả những cấu hình đã đo. Chín lượt mỗi cấu hình mô tả sai số trên các seed đã đo. Kết luận về Big-O cần dựa thêm vào phân tích chi phí thuật toán; không suy rộng từ các điểm đo hữu hạn.